In [ ]:
!pip install --upgrade pip
!pip install pandas==3.0.2

In [ ]:
# импорт и первые таблицы
import pandas as pd
import numpy as np

clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4],
    'name': ['Иван', 'Ольга', 'Дмитрий', 'Екатерина'],
    'city': ['Москва', 'СПБ', 'Москва', 'Казань']
})
loans = pd.DataFrame({
    'client_id': [2, 3, 4, 5],
    'amount': [500000, 300000, 200000, 100000],
    'term_months': [12, 24, 36, 12]
})
print("Clients:", clients, sep='\n')
print("\nLoans:", loans, sep='\n')

Clients:
   client_id       name    city
0          1       Иван  Москва
1          2      Ольга     СПБ
2          3    Дмитрий  Москва
3          4  Екатерина  Казань

Loans:
   client_id  amount  term_months
0          2  500000           12
1          3  300000           24
2          4  200000           36
3          5  100000           12


In [ ]:
# inner merge
inner = pd.merge(clients, loans, on='client_id', how='inner')
print("Inner merge:\n", inner)

Inner merge:
    client_id       name    city  amount  term_months
0          2      Ольга     СПБ  500000           12
1          3    Дмитрий  Москва  300000           24
2          4  Екатерина  Казань  200000           36


In [ ]:
# left и right merge
left = pd.merge(clients, loans, on='client_id', how='left')
print("Left merge:\n", left)
right = pd.merge(clients, loans, on='client_id', how='right')
print("\nRight merge:\n", right)

Left merge:
    client_id       name    city    amount  term_months
0          1       Иван  Москва       NaN          NaN
1          2      Ольга     СПБ  500000.0         12.0
2          3    Дмитрий  Москва  300000.0         24.0
3          4  Екатерина  Казань  200000.0         36.0

Right merge:
    client_id       name    city  amount  term_months
0          2      Ольга     СПБ  500000           12
1          3    Дмитрий  Москва  300000           24
2          4  Екатерина  Казань  200000           36
3          5        NaN     NaN  100000           12


In [ ]:
#  outer merge
outer = pd.merge(clients, loans, on='client_id', how='outer')
print("Outer merge:\n", outer)

Outer merge:
    client_id       name    city    amount  term_months
0          1       Иван  Москва       NaN          NaN
1          2      Ольга     СПБ  500000.0         12.0
2          3    Дмитрий  Москва  300000.0         24.0
3          4  Екатерина  Казань  200000.0         36.0
4          5        NaN     NaN  100000.0         12.0


In [ ]:
# indicator
ind = pd.merge(clients, loans, on='client_id', how='outer', indicator=True)
print("With indicator:\n", ind)
print("\nCounts by _merge:\n", ind['_merge'].value_counts())

With indicator:
    client_id       name    city    amount  term_months      _merge
0          1       Иван  Москва       NaN          NaN   left_only
1          2      Ольга     СПБ  500000.0         12.0        both
2          3    Дмитрий  Москва  300000.0         24.0        both
3          4  Екатерина  Казань  200000.0         36.0        both
4          5        NaN     NaN  100000.0         12.0  right_only

Counts by _merge:
 _merge
both          3
left_only     1
right_only    1
Name: count, dtype: int64


In [ ]:
# join по индексу
cl_i = clients.set_index('client_id')
lo_i = loans.set_index('client_id')
joined = cl_i.join(lo_i, how='inner')
print("Join on index:\n", joined)

Join on index:
                 name    city  amount  term_months
client_id                                        
2              Ольга     СПБ  500000           12
3            Дмитрий  Москва  300000           24
4          Екатерина  Казань  200000           36


In [ ]:
# concat строк (axis=0)
new_clients = pd.DataFrame({
    'client_id': [5, 6],
    'name': ['Сергей', 'Анна'],
    'city': ['Казань', 'СПБ']
})
all_clients = pd.concat([clients, new_clients], axis=0, ignore_index=True)
print("All clients (concat rows):\n", all_clients)

All clients (concat rows):
    client_id       name    city
0          1       Иван  Москва
1          2      Ольга     СПБ
2          3    Дмитрий  Москва
3          4  Екатерина  Казань
4          5     Сергей  Казань
5          6       Анна     СПБ


In [ ]:
# concat колонок (axis=1)
details = pd.DataFrame({
    'age': [30, 25, 40, 22, 35, 28],
    'income': [80000, 60000, 120000, 50000, 90000, 75000]
})
combined = pd.concat([all_clients, details], axis=1)
print("Combined (concat columns):\n", combined)

Combined (concat columns):
    client_id       name    city  age  income
0          1       Иван  Москва   30   80000
1          2      Ольга     СПБ   25   60000
2          3    Дмитрий  Москва   40  120000
3          4  Екатерина  Казань   22   50000
4          5     Сергей  Казань   35   90000
5          6       Анна     СПБ   28   75000


In [ ]:
# полный профиль
defaults = pd.DataFrame({
    'client_id': [1, 3, 5],
    'has_default': [True, False, True],
    'default_date': ['2023-05-10', None, '2024-01-15']
})
profile = (clients.merge(loans, on='client_id', how='left')
                 .merge(defaults, on='client_id', how='left'))
print("Full client profile:\n", profile)

Full client profile:
    client_id       name    city    amount  term_months has_default  \
0          1       Иван  Москва       NaN          NaN        True   
1          2      Ольга     СПБ  500000.0         12.0         NaN   
2          3    Дмитрий  Москва  300000.0         24.0       False   
3          4  Екатерина  Казань  200000.0         36.0         NaN   

  default_date  
0   2023-05-10  
1          NaN  
2          NaN  
3          NaN  


In [ ]:
# suffixes при одинаковых колонках
loans2 = loans.rename(columns={'amount': 'amount2'})
m = pd.merge(clients, loans2, on='client_id', how='left')
# если бы колонка 'amount' была в обеих, сработали бы _x и _y
print("Merge with renamed column:\n", m)

Merge with renamed column:
    client_id       name    city   amount2  term_months
0          1       Иван  Москва       NaN          NaN
1          2      Ольга     СПБ  500000.0         12.0
2          3    Дмитрий  Москва  300000.0         24.0
3          4  Екатерина  Казань  200000.0         36.0


Создать все возможные комбинации клиентов и кредитных продуктов.

In [ ]:
# Подготовим таблицы
clients = pd.DataFrame({'client_id': [1, 2], 'name': ['Иван', 'Ольга']})
products = pd.DataFrame({'product': ['Кредит наличными', 'Ипотека'], 'rate': [12.5, 7.0]})

# cross join
cross = pd.merge(clients, products, how='cross')
print(cross)

   client_id   name           product  rate
0          1   Иван  Кредит наличными  12.5
1          1   Иван           Ипотека   7.0
2          2  Ольга  Кредит наличными  12.5
3          2  Ольга           Ипотека   7.0


Найти клиентов, у которых нет кредитов (анти-соединение)

In [ ]:
# Клиенты и их заявки
clients = pd.DataFrame({'client_id': [1, 2, 3, 4], 'name': ['Иван', 'Ольга', 'Петр', 'Анна']})
loans = pd.DataFrame({'client_id': [1, 3], 'amount': [500000, 300000]})

# left_anti — исключаем клиентов, у которых есть кредиты
no_loans = pd.merge(clients, loans, on='client_id', how='left_anti')
print(no_loans)

   client_id   name  amount
1          2  Ольга     NaN
3          4   Анна     NaN


Найти кредиты, для которых по какой-то причине нет информации о клиенте

In [ ]:
# right_anti: оставляем только строки из правой таблицы, которых нет в левой
orphan_loans = pd.merge(clients, loans, on='client_id', how='right_anti')
print(orphan_loans)
# Если все client_id из loans есть в clients, результат будет пустым

Empty DataFrame
Columns: [client_id, name, amount]
Index: []


Использовать join() с параметром on, как альтернативу merge()

In [ ]:
clients.set_index('client_id', inplace=True)

# join с указанием колонки для связи (работает как merge, но через индекс)
joined = clients.join(loans.set_index('client_id'), how='inner')
print(joined)

# вариант без переноса индекса: указать on в правой таблице
clients.reset_index(inplace=True)
joined2 = clients.join(loans.set_index('client_id'), on='client_id', how='left')
print(joined2)

           name  amount
client_id              
1          Иван  500000
3          Петр  300000
   client_id   name    amount
0          1   Иван  500000.0
1          2  Ольга       NaN
2          3   Петр  300000.0
3          4   Анна       NaN


Проверить, что ключи для merge уникальны (one_to_one)

In [ ]:
# merge с проверкой уникальности ключей
try:
    pd.merge(clients, loans, on='client_id', how='left', validate='one_to_one')
except pd.errors.MergeError as e:
    print(f"Ошибка: {e}")

Сделать merge и явно задать суффиксы для колонок с одинаковыми именами

In [ ]:
df1 = pd.DataFrame({'id': [1, 2], 'val': [10, 20], 'note': ['a', 'b']})
df2 = pd.DataFrame({'id': [1, 3], 'val': [30, 40], 'note': ['c', 'd']})

result = pd.merge(df1, df2, on='id', how='outer', suffixes=('_left', '_right'))
print(result)

   id  val_left note_left  val_right note_right
0   1      10.0         a       30.0          c
1   2      20.0         b        NaN        NaN
2   3       NaN       NaN       40.0          d


Склеить несколько таблиц и пометить источник каждой строки

In [ ]:
df_2023 = pd.DataFrame({'client_id': [1, 2], 'revenue': [100, 200]})
df_2024 = pd.DataFrame({'client_id': [2, 3], 'revenue': [250, 150]})

# concat с ключами
combined = pd.concat([df_2023, df_2024], axis=0, keys=['2023', '2024'])
print(combined)

        client_id  revenue
2023 0          1      100
     1          2      200
2024 0          2      250
     1          3      150
